In [1]:
import pandas as pd
import numpy as np
import json
import seaborn as sns
import matplotlib.pyplot as plt

# Warning off
pd.options.mode.chained_assignment = None

import warnings
warnings.filterwarnings("ignore")

In [24]:
def export_results(df, narcissus = False, header=None, filters = {},cinic=False):

    # Iterate through the filters
    for key, value in filters.items():
        df = df[df[key] == value]

    # Initialize an empty DataFrame for results
    if narcissus:
        if header is None:
            columns = ['Avg Poison Success', 'Avg Nat Acc','Max Poison Success']
            if cinic: columns += ['Cifar-10 Nat Acc']
        else:
            columns = pd.MultiIndex.from_product([np.sort(df[header].unique()), ['Avg Poison Success', 'Avg Nat Acc','Max Poison Success']],
                                     names=[header, 'Metric'])
            if cinic:
                columns = pd.MultiIndex.from_product([np.sort(df[header].unique()), ['Avg Poison Success', 'Avg Nat Acc','Max Poison Success','Cifar-10 Nat Acc']],
                                     names=[header, 'Metric'])

    else:
        if header is None:
            columns = ['Poison Success', 'Natural Accuracy']
        else:
            columns = pd.MultiIndex.from_product([df[header].unique(), ['Poison Success', 'Natural Accuracy']],
                                                names=[header, 'Metric'])

    # Define the defenses
    defenses = ['None', 'EBM','Diff','EBM_Diff']                                         
    
    # Populate the DataFrame
    results_df = pd.DataFrame(columns=columns, index=defenses)

    if header is None: header_idxs = [0]
    else: header_idxs = df[header].unique()

    for header_idx in header_idxs:

        for defense in defenses:
            if header is None:
                df_def = df[(df['Defense'] == defense)]
            else:
                df_def = df[(df['Defense'] == 'None') & (df[header] == header_idx)]

            print(f'Added {defense} to results_df with num rows {len(df_def)}')

            if narcissus:
                avg_poison_success = df_def['P1 Acc'].mean() * 100
                std_poison_success = df_def['P1 Acc'].std() * 100
                avg_nat_acc = df_def['End Acc'].mean()
                std_nat_acc = df_def['End Acc'].std()
                max_poison_success = df_def['P1 Acc'].max() * 100

                if cinic:
                    avg_nat_acc_cinic = df_def['cifar_acc'].mean()
                    std_nat_acc_cinic = df_def['cifar_acc'].std() * 100


                if header is None:
                    results_df.loc[defense, 'Avg Poison Success'] = f'{avg_poison_success:.2f}\u00B1{std_poison_success:.1f}'
                    results_df.loc[defense,'Avg Nat Acc'] = f'{avg_nat_acc:.2f}\u00B1{std_nat_acc:.2f}'
                    results_df.loc[defense,  'Max Poison Success'] = f'{max_poison_success:.2f}'

                    if cinic:
                        results_df.loc[defense, 'Cifar-10 Nat Acc'] = f'{avg_nat_acc_cinic:.2%}\u00B1{std_nat_acc_cinic:.1f}'

                else:
                    results_df.loc[defense, (header_idx, 'Avg Poison Success')] = f'{avg_poison_success:.2%}\u00B1{std_poison_success:.1f}'
                    results_df.loc[defense, (header_idx, 'Avg Nat Acc')] = f'{avg_nat_acc:.2f}\u00B1{std_nat_acc:.1f}'
                    results_df.loc[defense, (header_idx, 'Max Poison Success')] = f'{max_poison_success:.2f}'

                    if cinic:
                        results_df.loc[defense, (header_idx, 'Cifar-10 Nat Acc')] = f'{avg_nat_acc_cinic:.2%}\u00B1{std_nat_acc_cinic:.1f}'

            else:

                success_rate = (df_def["Success"] == True).sum() / len(df_def)
                avg_nat_acc = df_def["End Acc"].mean()
                std_nat_acc = df_def["End Acc"].std()

                if header is None:
                    results_df.loc[defense, 'Poison Success'] = f'{success_rate:.2%}'
                    results_df.loc[defense, 'Natural Accuracy'] = f'{avg_nat_acc:.2f}%\u00B1{std_nat_acc:.1f}'

                else:
                    results_df.loc[defense, (header_idx, 'Poison Success')] = f'{success_rate:.2%}'
                    results_df.loc[defense, (header_idx, 'Natural Accuracy')] = f'{avg_nat_acc:.2f}%\u00B1{std_nat_acc:.1f}'

    return results_df

In [25]:
df = pd.read_csv('/home/sunaybhat/results_EBM_Defense/From_Scratch/Narcissus/Results.csv')

df['Defense'] = df['Defense'].fillna('None')

df_results = export_results(df,narcissus=True)

df_results

Added None to results_df with num rows 10
Added EBM to results_df with num rows 10
Added Diff to results_df with num rows 10
Added EBM_Diff to results_df with num rows 10


,Avg Poison Success,Avg Nat Acc,Max Poison Success
None,43.95±33.6,94.89±0.23,93.59
EBM,1.34±0.6,92.73±0.22,2.29
Diff,2.00±2.0,93.50±0.16,7.49
EBM_Diff,1.75±1.3,92.38±0.26,5.19
